# BamiBERT LoRA với trọng số dương 2

Huấn luyện lại bộ trích xuất với trọng số dương 2. Bộ phân loại hồ sơ tái sử dụng từ notebook 01.

[Báo cáo tổng hợp](reports/tong-hop-pipeline-smartnotice-v3.3-2026-11-23.md) · [Giao thức đánh giá](../../docs/evaluation.md)

## 1. Cấu hình và dữ liệu

Nạp thông báo đã chuẩn bị và kiểm tra hash, nhóm, schema và các khoảng ký tự. Tham chiếu test v3.3 là chú giải lịch sử có máy hỗ trợ.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

repo = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tools/compare_notice_pipelines.py").is_file()),
    None,
)
if repo is None:
    raise RuntimeError("Mở notebook từ workspace SmartNotice.")
os.chdir(repo)
split = Path(os.environ.get("SMARTNOTICE_SPLIT_ROOT", "tmp/drive-run-20261123-fresh")).resolve()
base = Path(os.environ.get("SMARTNOTICE_BAMIBERT_DIR", "tmp/bamibert-checkpoint")).resolve()
destination = Path(os.environ.get(
    "SMARTNOTICE_NOTEBOOK_OUTPUT", "reports/generated/pipeline-notebooks-v33-20261123"
)).resolve()
baseline = Path(os.environ.get(
    "SMARTNOTICE_BASELINE", "reports/generated/bamibert/v3_3-2026-11-23-final-seed42"
)).resolve()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def run_tool(script, *arguments, **environment):
    env = {**os.environ, "PYTHONHASHSEED": "42", "TOKENIZERS_PARALLELISM": "false", **environment}
    result = subprocess.run(
        [sys.executable, "-u", str(repo / "tools" / script), *map(str, arguments)],
        cwd=repo, env=env, capture_output=True, text=True, encoding="utf-8", errors="replace",
    )
    if result.returncode:
        print(result.stdout[-4000:], result.stderr[-4000:])
        result.check_returncode()
    print(result.stdout.strip())

In [ ]:
from smartnotice.modeling.data import labeled_links, load_development, sha256, span_key

development, provenance = load_development(split)
for part, docs in development.items():
    print(part, {
        "notices": len(docs),
        "groups": len({d.group_id for d in docs}),
        "typed_spans": sum(len({span_key(s) for s in d.annotation.spans}) for d in docs),
        "links": sum(len(set(labeled_links(d.annotation))) for d in docs),
        "events": sum(len(d.annotation.events) for d in docs),
    })
print("Phân bổ:", provenance["split_counts"], "gold_ready:", provenance["gold_ready"])

## 2. Token hóa và huấn luyện bộ trích xuất

In [ ]:
os.environ["SMARTNOTICE_SPLIT_ROOT"] = str(split)
os.environ["SMARTNOTICE_RUN_DIR"] = str(destination / "run-03")
os.environ["SMARTNOTICE_BAMIBERT_DIR"] = str(base)
os.environ["SMARTNOTICE_TUNING"] = "lora"
os.environ["SMARTNOTICE_SPAN_NMS"] = "0"
os.environ.pop("SMARTNOTICE_RESUME_EXTRACTION", None)
import gc
import hashlib
import json
import os
import platform
import shutil
from dataclasses import asdict
from importlib.metadata import version
from importlib.resources import files
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from smartnotice.modeling.applicability import train_applicability
from smartnotice.modeling.data import choose_width, load_development, sha256, tokenize
from smartnotice.modeling.experiment import (
    Settings,
    evaluate_extraction,
    load_extractor,
    train_extraction,
    write_summary,
)
from transformers import AutoTokenizer

MODEL = "Qualcomm-AI-Research/BamiBERT"
REVISION = "57bc1340debbe4e348ec549047a763caebe4a977"
MODEL_SHA256 = "c8c8abe5914f2543bd6a65019b685dacf25ad39da5f01b680b082876e1947627"
root = split
output = destination / "run-03"
output.mkdir(parents=True, exist_ok=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vram = torch.cuda.get_device_properties(device).total_memory if device.type == "cuda" else 0
settings = Settings(
    tuning=os.environ.get("SMARTNOTICE_TUNING", "lora" if vram < 7 * 2**30 else "full"),
    suppress_same_type_overlap=os.environ.get("SMARTNOTICE_SPAN_NMS") == "1",
)
print(device, flush=True)
checkpoint = os.environ.get("SMARTNOTICE_BAMIBERT_DIR") or snapshot_download(
    MODEL,
    revision=REVISION,
    allow_patterns=["*.json", "*.safetensors"],
)
checkpoint_path = Path(checkpoint)
weights_sha256 = sha256(checkpoint_path / "model.safetensors")
if weights_sha256 != MODEL_SHA256:
    raise ValueError("BamiBERT weights do not match the pinned release")
tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
assert tokenizer.is_fast
print(tokenizer.__class__.__name__)
from dataclasses import replace
settings = replace(settings, span_positive_weight=2.0)

In [ ]:
documents, provenance = load_development(root)
width = choose_width(documents["train"], tokenizer, settings.length, settings.overlap)
tokenized = {
    split: [tokenize(doc, tokenizer, settings.length, settings.overlap, width) for doc in items]
    for split, items in documents.items()
}
run = {
    "model": MODEL,
    "revision": REVISION,
    "settings": asdict(settings),
    "model_weights_sha256": weights_sha256,
    "tokenizer_sha256": sha256(checkpoint_path / "tokenizer.json"),
    "tokenizer_class": tokenizer.__class__.__name__,
    "data": provenance,
    "max_width_subwords": width,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "device": str(device),
    "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
    "versions": {name: version(name) for name in ("torch", "transformers", "scikit-learn")},
    "peft_version": version("peft") if settings.tuning == "lora" else None,
    "code_sha256": {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest()
        for p in files("smartnotice.modeling").iterdir()
        if p.name.endswith(".py")
    },
    "label_scope": "development only",
    "counts": {
        split: {
            "documents": len(items),
            "windows": sum(len(d.windows) for d in items),
            "groups": len({d.document.group_id for d in items}),
        }
        for split, items in tokenized.items()
    },
}
write_summary(output / "run.json", run)
print(run["counts"], "max width:", width)

In [ ]:
from dataclasses import asdict

reuse = os.environ.get("SMARTNOTICE_FROZEN_EXTRACTION")
if reuse:
    prior = Path(reuse).resolve()
    previous = read_json(prior / "run.json")
    for field in ("data", "model_weights_sha256", "tokenizer_sha256", "max_width_subwords"):
        if previous[field] != run[field]:
            raise ValueError(f"Checkpoint metadata differs: {field}")
    if Settings(**previous["settings"]) != settings:
        raise ValueError("Checkpoint training settings differ")
    shutil.copyfile(prior / "extraction.pt", output / "extraction.pt")
    extraction = read_json(prior / "extraction_metrics.json")
    write_summary(output / "extraction_metrics.json", extraction)
    run["frozen_extraction"] = {
        "source": str(prior.relative_to(repo)) if prior.is_relative_to(repo) else prior.name,
        "sha256": sha256(output / "extraction.pt"),
        "settings": asdict(settings),
    }
    write_summary(output / "run.json", run)
    print("Nap trong so trich xuat da huan luyen, SHA256", run["frozen_extraction"]["sha256"])
    print("Nguong da chot tren validation:", extraction["thresholds"])
else:
    extraction, validation_predictions = train_extraction(
        tokenized["train"], tokenized["validation"], checkpoint, tokenizer.pad_token_id,
        width, settings, output, device,
    )
    print("Ngưỡng validation:", extraction["thresholds"])
    print("F1 validation:", extraction["span"]["micro"]["f1"], extraction["link_predicted_endpoints"]["micro"]["f1"])
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

## 3. Suy luận test và chấm điểm

In [ ]:
from smartnotice.modeling.experiment import write_summary
from smartnotice.modeling.metrics import prf


def show(scores):
    span = scores["span"]["micro"]
    link = scores["link"]["micro"]
    event = scores["event"]
    for caption, score in (("Cụm có kiểu", span), ("Quan hệ", link), ("Sự kiện", event)):
        print(
            f"{caption}: F1 {score['f1']:.2%} | P {score['precision']:.2%} | "
            f"R {score['recall']:.2%} | đúng/dự đoán/tham chiếu "
            f"{score['matched']}/{score['predicted']}/{score['expected']}"
        )
    print("Ngày tường minh:", scores["dates"]["explicit"])


def without_reference(docs):
    from smartnotice.modeling.data import Document
    from smartnotice.schemas import Annotation

    return [
        Document(d.notice, Annotation(
            doc_id=d.notice.doc_id, annotator_id="inference", spans=[], events=[]
        ), d.group_id)
        for d in docs
    ]


def boundary_diagnostics(docs, predictions):
    expected, predicted = {}, {}
    for d in docs:
        doc_id = d.notice.doc_id
        for target, spans in (
            (expected, map(span_key, d.annotation.spans)), (predicted, predictions[doc_id][0])
        ):
            for begin, end, kind in spans:
                target.setdefault((doc_id, begin, end), set()).add(kind)
    matched = expected.keys() & predicted.keys()
    return {
        "untyped": prf(set(expected), set(predicted)),
        "correct_type_sets": sum(expected[key] == predicted[key] for key in matched),
        "matched_boundaries": len(matched),
    }

In [ ]:
import torch
from transformers import AutoTokenizer
from tools.compare_notice_pipelines import _bert_predictions

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(base, use_fast=True)

In [ ]:
from tools.compare_notice_pipelines import _metrics, _test_docs

test = _test_docs(split)
print("Test:", len(test), "thông báo,", len({d.group_id for d in test}), "nhóm")

In [ ]:
out = output
weights = out / "extraction.pt"
app_run = baseline

record = read_json(app_run / "run.json")
assert record["data"]["hashes"] == provenance["hashes"]
assert record["model_weights_sha256"] == sha256(base / "model.safetensors")
assert record["tokenizer_sha256"] == sha256(base / "tokenizer.json")

predictions = _bert_predictions(weights, without_reference(test), base, tokenizer, device)
result = _metrics(test, predictions)
result["boundaries"] = boundary_diagnostics(test, predictions)
show(result)
print("Ranh giới và gán kiểu có điều kiện:", result["boundaries"])
write_summary(out / "test_scores.json", result)

In [ ]:
if os.environ.get("SMARTNOTICE_SKIP_APPLICABILITY") == "1":
    print("Applicability evaluation disabled for this run.")
else:
    from types import SimpleNamespace
    from tools.compare_notice_pipelines import _app

    app = _app(
        SimpleNamespace(base=base, baseline=app_run),
        development["train"], test, {"pipeline": predictions}, device, tokenizer,
    )["pipeline"]
    result["applicability"] = app
    print("F1 sự kiện và nhãn áp dụng:", f"{app['joint_exact_event_profile']['f1']:.2%}")
    write_summary(out / "test_scores.json", result)